# Day 14 Revision Summary — Overfitting, Cross-Validation & Data Leakage

- **Overfitting vs underfitting**: underfitting = low train AND test score (too simple); overfitting = train score much higher than test score (memorised noise, not the pattern) — the tell is the **gap** between train and test.
- A `DecisionTreeClassifier` with `max_depth=None` hits train score 1.000 (perfect memorisation) while test score actually drops — depth 3 gave the best test score and the smallest gap on the breast-cancer data.
- **Cross-validation** (`cross_val_score(model, X, y, cv=5)`) splits data 5 ways, trains/tests 5 times, and reports **mean ± std** instead of one lucky/unlucky single split (e.g. `0.951 ± 0.018`).
- **Data leakage**: any step that *learns* from data (scaling, feature selection, imputation) leaks if it runs on the full dataset before the split — it manufactured **86% "accuracy" from pure random noise** in the demo (honest score ≈ 0.50 / 0.415).
- The cure is a scikit-learn **`Pipeline`**: chain preprocessing + model so every learning step is refit per-fold, inside cross-validation — leakage becomes structurally impossible.

*Resource note: a top-level Google Doc "Day 14 — Overfitting, CV & Leakage · Formulas (First Principles)" and `resources/cheatsheet.md` cover the score/gap/mean/std/z-score formulas in full first-principles detail — referenced here, not reproduced.*

## Classwork Exercise 1 — Make It Overfit (`overfit_underfit.py`, ~12 min)

**What's being asked:** Fit a `DecisionTreeClassifier` on `breast_cancer` at depths 1, 2, 3, 5, and `None`. Print the train and test score at each depth, watch the train score climb to 1.000 while the test score peaks then falls, and name the best depth (highest test score, smallest gap). Verified reference numbers: depth 1 → train 0.923/test 0.921 (underfit); depth 3 → train 0.976/test 0.939 (good balance); depth 5 → train 0.993/test 0.921 (overfitting); depth `None` → train 1.000/test 0.912 (memorised).

**Approach:**
1. Load `load_breast_cancer(return_X_y=True)` and split with `train_test_split(..., test_size=0.2, random_state=42, stratify=y)`.
2. Loop over `depth in [1, 3, 5, None]` (add 2 too if you like).
3. Inside the loop, create `DecisionTreeClassifier(max_depth=depth, random_state=42)` and `.fit()` it on the training data.
4. Compute `model.score(X_train, y_train)` and `model.score(X_test, y_test)`.
5. Print `depth, train_score, test_score` (rounded) for every depth.
6. Identify and print which depth has the highest test score and the smallest train-test gap.

In [ ]:
"""
Day 14 · Classwork Exercise 1 -- Overfitting vs underfitting
Reference: depth 1 -> ~0.923/0.921 (underfit) ; depth 3 -> ~0.976/0.939 (good) ;
depth 5 -> ~0.993/0.921 (overfit) ; depth None -> ~1.000/0.912 (memorised)
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# TODO 1: load X, y and split (test_size=0.2, random_state=42, stratify=y)
X, y = None, None

# TODO 2: loop over these depths
for depth in [1, 3, 5, None]:
    # TODO 3: create DecisionTreeClassifier(max_depth=depth, random_state=42) and fit it
    model = None

    # TODO 4: compute train_score = model.score(X_train, y_train)
    #         and test_score = model.score(X_test, y_test)
    train_score = None
    test_score = None

    # TODO 5: print depth, train_score, test_score (rounded to 3 dp)

# TODO 6: after the loop, state which depth you'd pick and why (smallest gap,
# highest test score) -- print it or write a comment


## Classwork Exercise 2 — Cross-Validate (`cross_validation.py`, ~8 min)

**What's being asked:** Run `cross_val_score(model, X, y, cv=5)` on `breast_cancer` with a `LogisticRegression`, print the five fold scores, then print the mean and standard deviation, reporting the result as `mean ± std`. Verified expected scores: `[0.939, 0.947, 0.982, 0.930, 0.956]` → mean ≈ 0.951, std ≈ 0.018.

**Approach:**
1. Load `load_breast_cancer(return_X_y=True)`.
2. Create `LogisticRegression(max_iter=5000)` (no manual train/test split needed — CV does it).
3. Call `cross_val_score(model, X, y, cv=5)` to get 5 scores.
4. Print the individual fold scores (rounded).
5. Compute and print `scores.mean()` and `scores.std()`.
6. Print the final result formatted as `mean +/- std`, and note how much the folds differ from each other.

In [ ]:
"""
Day 14 · Exercise 2 -- Cross-validation
One split can be lucky or unlucky. Cross-validation splits FIVE ways
and averages, so your score doesn't depend on one random draw.
Expected: 5 fold scores ~[0.939, 0.947, 0.982, 0.930, 0.956] -> mean ~0.951, std ~0.018
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# TODO 1: load X, y from load_breast_cancer(return_X_y=True)
X, y = None, None

# TODO 2: create a LogisticRegression(max_iter=5000)
model = None

# TODO 3: run 5-fold CV: scores = cross_val_score(model, X, y, cv=5)
scores = None

# TODO 4: print the individual fold scores, rounded to 3 dp

# TODO 5: compute and print scores.mean() and scores.std()

# TODO 6: print the final line as "mean +/- std", e.g. f"-> {mean:.3f} +/- {std:.3f}"


## Classwork Exercise 3 — Catch the Leak (`leakage_pipeline.py`, ~18 min)

**What's being asked:** Build pure-noise data (random features, random 0/1 labels — no real pattern exists, so honest accuracy must be ~0.50). Compare a **leaky** approach (select features on the whole dataset, then cross-validate) against an **honest** approach (feature selection inside a `Pipeline`, cross-validated). Verified reference: leaky ≈ 0.860 (fake skill), honest ≈ 0.415 (the truth, near chance).

**Approach:**
1. Generate random data: e.g. `X = np.random.randn(10000, n_features)`, `y = np.random.randint(0, 2, 10000)`.
2. **Leaky version:** call `SelectKBest(f_classif, k=20).fit_transform(X, y)` on the *entire* dataset first, then `cross_val_score(LogisticRegression(...), X_selected, y, cv=5)`.
3. **Honest version:** build a `Pipeline([("select", SelectKBest(f_classif, k=20)), ("clf", LogisticRegression(...))])` and call `cross_val_score(pipe, X, y, cv=5)` directly on the raw `X`.
4. Print both mean scores side by side and note the size of the gap.
5. Write a one-line takeaway: any step that calls `.fit()` on data belongs *inside* the Pipeline.

In [ ]:
"""
Day 14 · Classwork Exercise 3 -- Catch the leak
Expected: LEAKY mean ~0.860 (fake skill) ; HONEST mean ~0.415 (near chance, the truth)
"""
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# TODO 1: build pure-noise data -- random features, random 0/1 labels, no real pattern
rng = np.random.default_rng(42)
X = None  # rng.standard_normal((10000, 50)) or similar
y = None  # rng.integers(0, 2, size=10000)

# TODO 2: LEAKY -- select features using ALL of X, y, THEN cross-validate
X_selected = None  # SelectKBest(f_classif, k=20).fit_transform(X, y)
leaky_scores = None  # cross_val_score(LogisticRegression(max_iter=1000), X_selected, y, cv=5)

# TODO 3: HONEST -- put selection INSIDE a Pipeline, cross-validate on raw X
pipe = None  # Pipeline([("select", SelectKBest(f_classif, k=20)),
             #           ("clf", LogisticRegression(max_iter=1000))])
honest_scores = None  # cross_val_score(pipe, X, y, cv=5)

# TODO 4: print leaky_scores.mean() vs honest_scores.mean() -- how big is the gap?

# TODO 5: write a one-line takeaway comment about where preprocessing steps belong


## Homework Exercise 1 — KNN Overfitting Sweep (~20 min)

**What's being asked:** Rebuild the Exercise 1 overfitting table, but for a `KNeighborsClassifier` as `k` goes from 1 to 50. Find the sweet spot (the `k` with the best test score / smallest train-test gap).

**Approach:**
1. Reuse the `breast_cancer` train/test split from Exercise 1.
2. Loop `k` over a range, e.g. `range(1, 51)`.
3. For each `k`, fit a `KNeighborsClassifier(n_neighbors=k)` and record train and test scores.
4. Store results (e.g. in a list of tuples or a small DataFrame) so you can compare across `k`.
5. Print or plot train vs test score across `k` and identify where the gap is smallest while test score stays high — that's the sweet spot.
6. Note: unlike a single decision tree, small `k` (e.g. 1) tends to overfit here, while large `k` tends to underfit — the opposite direction from tree depth.

In [ ]:
"""
Day 14 · Homework 1 -- KNN overfitting sweep, k = 1..50
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = load_breast_cancer(return_X_y=True)
# TODO 1: split (test_size=0.2, random_state=42, stratify=y) -- reuse Exercise 1's split logic

results = []
# TODO 2: loop k from 1 to 50
for k in range(1, 51):
    # TODO 3: fit KNeighborsClassifier(n_neighbors=k) on the training data
    knn = None

    # TODO 4: record (k, train_score, test_score) into results
    train_score = None
    test_score = None
    # results.append((k, train_score, test_score))

# TODO 5: print or plot results -- find the k with the best test score / smallest gap

# TODO 6: write a one-line comment on the sweet-spot k you found


## Homework Exercise 2 — Wrap a Pipeline (~15 min)

**What's being asked:** Wrap a `StandardScaler` + `LogisticRegression` in a `Pipeline` and cross-validate it, mirroring the leakage-proof pattern from Classwork Exercise 3.

**Approach:**
1. Load `load_breast_cancer(return_X_y=True)`.
2. Build `Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])`.
3. Run `cross_val_score(pipe, X, y, cv=5)` on the raw (unscaled) `X` — the pipeline scales inside each fold automatically.
4. Print the fold scores, mean, and std.
5. Compare informally to the unscaled `LogisticRegression` cross-validation from Classwork Exercise 2 — does scaling help here?

In [ ]:
"""
Day 14 · Homework 2 -- StandardScaler + LogisticRegression in a Pipeline
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# TODO 1: load X, y
X, y = None, None

# TODO 2: build the Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(...))])
pipe = None

# TODO 3: cross_val_score(pipe, X, y, cv=5) on the RAW X (no manual scaling beforehand!)
scores = None

# TODO 4: print the fold scores, mean, and std

# TODO 5: compare to the unscaled cross_val_score mean from Classwork Exercise 2 (~0.951)
# -- did scaling change the result much? Print or comment on your observation.


## Other homework items (no code needed)

- **One paragraph:** Describe a real-world data leak in your own words (hint: using tomorrow's data to predict today, e.g. a feature that wouldn't actually be available at prediction time in production).
- **Commit your work:** `git add . && git commit -m "day 14"`.